# Panchatantra-Based Cross-Domain Generalization and Few-Shot Adaptation for Sanskrit POS Tagging

**M.Tech Dissertation Research**  
**Title:** *Cross-Domain Generalization and Few-Shot Adaptation for Sanskrit POS Tagging*  
**Source Domain:** `Panchatantra_clean.csv` (Classical Sanskrit Prose/Fables)  
**Target Domain:** `News_Annotated.csv` (Modern Sanskrit News Prose)  
**Reference-Only (Excluded):** `Literature_Corpus.csv`

## Dissertation Objectives
1. **Objective 1 (Cross-Domain Generalization):** Quantify the performance difference between in-domain Sanskrit POS tagging (Panchatantra) and zero-shot cross-domain POS tagging (Panchatantra → News) across five model architectures.
2. **Objective 2 (Few-Shot Adaptation):** Quantify to what extent few-shot fine-tuning with `[50, 100, 200, 500, 1000]` annotated news sentences improves cross-domain performance when independently initialized from each model's Panchatantra-trained checkpoint.

## Three-Stage Experimental Pipeline
- **Stage 1:** Panchatantra In-Domain Training & Held-Out Evaluation (70% Train / 15% Val / 15% Test)
- **Stage 2:** Zero-Shot Cross-Domain Evaluation (Panchatantra Checkpoint → Locked News Test)
- **Stage 3:** Few-Shot Cross-Domain Adaptation (Nested Budgets `50 ⊂ 100 ⊂ 200 ⊂ 500 ⊂ 1000`, Independent Initialization)

## 1. Environment and Dependency Checks

In [1]:
import os
import sys
import json
import copy
import time
import shutil
import random
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python Version      : {platform.python_version()}")
print(f"PyTorch Version     : {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory          : {props.total_memory / (1024**3):.2f} GB")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device    : {DEVICE}")

/home/siba/anaconda3/envs/muril_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python Version      : 3.12.12
PyTorch Version     : 2.10.0+cu128
Transformers Version: 5.2.0
CUDA Available      : True
GPU Device          : NVIDIA H100 NVL
GPU Memory          : 93.09 GB
Execution Device    : cuda


## 2. Project Path Configuration

In [2]:
# Resolve project root dynamically
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / "data" / "raw"
PANCHATANTRA_RAW = DATA_RAW / "Panchatantra_clean.csv"
NEWS_RAW = DATA_RAW / "News_Annotated.csv"
EXISTING_DATASET_DIR = PROJECT_ROOT / "outputs" / "datasets"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "Panchatantra_output"
DIRS = {
    "datasets": OUTPUT_ROOT / "datasets",
    "models": OUTPUT_ROOT / "models",
    "evaluations": OUTPUT_ROOT / "evaluations",
    "predictions": OUTPUT_ROOT / "predictions",
    "few_shot_samples": OUTPUT_ROOT / "few_shot_samples",
    "figures": OUTPUT_ROOT / "figures",
    "confusion_matrices": OUTPUT_ROOT / "figures" / "confusion_matrices",
    "logs": OUTPUT_ROOT / "logs",
    "configurations": OUTPUT_ROOT / "configurations",
    "reports": OUTPUT_ROOT / "reports",
    "experiment_status": OUTPUT_ROOT / "experiment_status",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
for m in ["bilstm_crf", "xlm_roberta", "indicbert", "muril", "sanskrit_bert"]:
    (DIRS["models"] / m).mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
FEW_SHOT_SEEDS = [42, 123, 2026]
FEW_SHOT_BUDGETS = [50, 100, 200, 500, 1000]

print("Project Root :", PROJECT_ROOT)
print("Output Root  :", OUTPUT_ROOT)
print("Source File  :", PANCHATANTRA_RAW, "| Exists:", PANCHATANTRA_RAW.is_file())
print("Target File  :", NEWS_RAW, "| Exists:", NEWS_RAW.is_file())

Project Root : /home/siba/P25DS023/Cross-Domain-23
Output Root  : /home/siba/P25DS023/Cross-Domain-23/outputs/Panchatantra_output
Source File  : /home/siba/P25DS023/Cross-Domain-23/data/raw/Panchatantra_clean.csv | Exists: True
Target File  : /home/siba/P25DS023/Cross-Domain-23/data/raw/News_Annotated.csv | Exists: True


## 3. Dataset Loading

In [3]:
from src.data_preparation.dataset import (
    TARGET_TAGS, TAG2ID, ID2TAG,
    validate_split_frame, validate_split_leakage, get_nested_few_shot_subsets
)
from src.models.bilstm_crf import BiLSTMCRF, SequenceTaggingDataset, build_vocabulary, collate_sequences
from src.models.transformer_tagger import TransformerTagger, devanagari_to_iast
from src.training.trainer import train_bilstm_crf, decode_bilstm_predictions, train_transformer
from src.evaluation.evaluator import evaluate_predictions, save_evaluation_results
from src.evaluation.visualizer import plot_confusion_matrix_heatmap

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

raw_panch_df = pd.read_csv(PANCHATANTRA_RAW)
raw_news_df = pd.read_csv(NEWS_RAW)
print(f"Loaded Panchatantra_clean.csv : {len(raw_panch_df):,} token rows, {raw_panch_df['sentence_id'].nunique():,} sentence_ids")
print(f"Loaded News_Annotated.csv     : {len(raw_news_df):,} token rows, {raw_news_df['doc_id'].nunique():,} doc_ids")

Loaded Panchatantra_clean.csv : 43,653 token rows, 5,393 sentence_ids
Loaded News_Annotated.csv     : 24,299 token rows, 1,971 doc_ids


## 4. Dataset Inspection

In [4]:
print("=== Panchatantra_clean.csv Schema ===")
print("Columns:", raw_panch_df.columns.tolist())
print("Null counts:\n", raw_panch_df.isnull().sum())
print("\nUnique source tags:", raw_panch_df["tag"].nunique(dropna=True))
print(raw_panch_df["tag"].value_counts(dropna=False))

print("\n=== News_Annotated.csv Schema ===")
print("Columns:", raw_news_df.columns.tolist())
print("Null counts:\n", raw_news_df[["doc_id", "sentence_id", "token", "gold_sanskrit_tag"]].isnull().sum())
print("\nUnique target tags:", raw_news_df["gold_sanskrit_tag"].nunique())
print(raw_news_df["gold_sanskrit_tag"].value_counts())

=== Panchatantra_clean.csv Schema ===
Columns: ['sentence_id', 'word_number', 'word', 'lemma', 'tag', 'sentence']
Null counts:
 sentence_id        0
word_number        0
word               0
lemma          43653
tag                6
sentence           0
dtype: int64

Unique source tags: 28
tag
NP      9690
PUNC    9413
SN      4768
A       4334
KP      3854
KN      2225
NV      1958
KKS     1317
AVK      876
AN       846
NS       729
APY      541
KV       401
SNV      386
AP       365
AUD      348
INTF     305
KNV      278
ASMB     213
SSN      171
ASD      133
AUP      117
SMKY      97
ABS       87
DV        85
SMKP      56
SMK       53
NaN        6
UA         1
Name: count, dtype: int64

=== News_Annotated.csv Schema ===
Columns: ['doc_id', 'sentence_id', 'annotation_id', 'sentence_text', 'token_id', 'token', 'lemma', 'upos', 'stanza_upos', 'xpos', 'feats', 'gold_sanskrit_tag', 'annotation_status', 'annotation_note', 'verified_by', 'verified_at']
Null counts:
 doc_id               0


## 5. Data Cleaning
We inspect missing/unresolved tags (`NaN` and `UA`) and slash-tag punctuation artifacts (`word/TAG,`) in `Panchatantra_clean.csv`.
Following strict research integrity rules:
- We **never fabricate** missing POS labels.
- Sentences containing `NaN` or unresolved `UA` tags are transparently excluded and logged.

In [5]:
slash_artifacts = raw_panch_df[raw_panch_df["word"].astype(str).str.contains("/", na=False)]
nan_rows = raw_panch_df[raw_panch_df["tag"].isna()]
ua_rows = raw_panch_df[raw_panch_df["tag"] == "UA"]
excluded_sids = set(nan_rows["sentence_id"]).union(set(ua_rows["sentence_id"]))

print(f"Tokens with '/' artifact (preserved as-is with source tag): {len(slash_artifacts)}")
print(f"Tokens with NaN tag: {len(nan_rows)}")
print(f"Tokens with unresolved 'UA' tag: {len(ua_rows)}")
print(f"Sentences excluded due to NaN/UA tags: {len(excluded_sids)} ({sorted(excluded_sids)})")

excluded_df = raw_panch_df[raw_panch_df["sentence_id"].isin(excluded_sids)].copy()
excluded_df.to_csv(DIRS["datasets"] / "panchatantra_excluded_sentences.csv", index=False)

clean_panch_df = raw_panch_df[~raw_panch_df["sentence_id"].isin(excluded_sids)].copy()
clean_panch_df["word"] = clean_panch_df["word"].astype(str)
clean_panch_df["tag"] = clean_panch_df["tag"].astype(str)
print(f"Clean Panchatantra tokens: {len(clean_panch_df):,} across {clean_panch_df['sentence_id'].nunique():,} sentences")

Tokens with '/' artifact (preserved as-is with source tag): 113
Tokens with NaN tag: 6
Tokens with unresolved 'UA' tag: 1
Sentences excluded due to NaN/UA tags: 7 ([177, 2384, 2393, 3775, 4061, 4336, 4655])
Clean Panchatantra tokens: 43,568 across 5,386 sentences


## 6. POS-Tag Mapping (Panchatantra 28-Tag Inventory → 24 Target Tags)
`Panchatantra_clean.csv` uses a 28-tag Sanskrit tagset (`NP, PUNC, SN, A, KP, KN, NV, KKS, AVK, AN, NS, APY, KV, SNV, AP, AUD, INTF, KNV, ASMB, SSN, ASD, AUP, SMKY, ABS, DV, SMKP, SMK, UA`).
We map the 27 linguistically valid source tags to the 24 `TARGET_TAGS` aligned with `News_Annotated.csv`:
- `NP → N` (Common Noun), `NS → NS` (Proper Noun)
- `NV → A` (Nominal Adjective, matching `JJ.* → A` and `News_Annotated` `A`)
- `KN → NV` (Participles/Kridanta), `KKS → NV` (Absolutives Ktvā/Lyap, matching `News_Annotated` `NV`), `KNV → KNV`
- `KP → KP` (Finite Verb), `SN → SN`, `SNV → SNV`, `SSN → SN` (Relative/Correlative Pronoun)
- `PUNC → PU`, `A / AN / INTF / AUD / ASD / AP / APY / AVK → AVN` (Indeclinables/Avyaya)
- `AUP → UP` (Upapada/Postposition), `ABS / ASMB → AVU` (Interjection/Vocative)
- `KV → AVP` (Adverb), `DV → DVI` (Reduplicative), `SMK / SMKY → AVC` (Numerals), `SMKP → A` (Ordinal Numeral)
- `UA / NaN` → Unmapped (Excluded)

In [6]:
PANCHATANTRA_TO_TARGET = {
    "NP": "N",
    "NS": "NS",
    "NV": "A",
    "KN": "NV",
    "KKS": "NV",
    "KNV": "KNV",
    "KP": "KP",
    "SN": "SN",
    "SNV": "SNV",
    "SSN": "SN",
    "PUNC": "PU",
    "A": "AVN",
    "AN": "AVN",
    "INTF": "AVN",
    "AUD": "AVN",
    "ASD": "AVN",
    "AP": "AVN",
    "APY": "AVN",
    "AVK": "AVN",
    "AUP": "UP",
    "ABS": "AVU",
    "ASMB": "AVU",
    "KV": "AVP",
    "DV": "DVI",
    "SMK": "AVC",
    "SMKY": "AVC",
    "SMKP": "A",
}

# Verify all mapped targets are in TARGET_TAGS
assert set(PANCHATANTRA_TO_TARGET.values()).issubset(set(TARGET_TAGS))

clean_panch_df["target_tag"] = clean_panch_df["tag"].map(PANCHATANTRA_TO_TARGET)
assert clean_panch_df["target_tag"].isna().sum() == 0, "Unmapped tags remain!"

# Build mapping audit table across ALL raw Panchatantra tokens
audit_rows = []
for src_tag, count in raw_panch_df["tag"].fillna("MISSING_NAN").value_counts().items():
    tgt = PANCHATANTRA_TO_TARGET.get(src_tag, "UNMAPPED_EXCLUDED")
    status = "mapped" if src_tag in PANCHATANTRA_TO_TARGET else "excluded_unresolved"
    audit_rows.append({
        "source_tag": src_tag,
        "target_tag": tgt,
        "mapping_status": status,
        "token_count": int(count),
        "percentage": round(count / len(raw_panch_df) * 100, 4),
    })
mapping_audit_df = pd.DataFrame(audit_rows)
mapping_audit_df.to_csv(DIRS["evaluations"] / "panchatantra_mapping_audit.csv", index=False)

mapped_tok_count = int(mapping_audit_df[mapping_audit_df["mapping_status"] == "mapped"]["token_count"].sum())
coverage_pct = mapped_tok_count / len(raw_panch_df) * 100
print(f"Mapping Coverage: {mapped_tok_count:,} / {len(raw_panch_df):,} tokens ({coverage_pct:.2f}%)")
print(mapping_audit_df.to_string(index=False))

Mapping Coverage: 43,646 / 43,653 tokens (99.98%)
 source_tag        target_tag      mapping_status  token_count  percentage
         NP                 N              mapped         9690     22.1978
       PUNC                PU              mapped         9413     21.5632
         SN                SN              mapped         4768     10.9225
          A               AVN              mapped         4334      9.9283
         KP                KP              mapped         3854      8.8287
         KN                NV              mapped         2225      5.0970
         NV                 A              mapped         1958      4.4854
        KKS                NV              mapped         1317      3.0170
        AVK               AVN              mapped          876      2.0067
         AN               AVN              mapped          846      1.9380
         NS                NS              mapped          729      1.6700
        APY               AVN              mapped 

## 7. Dataset Statistics and Natural Class Imbalance Analysis

In [7]:
panch_tag_dist = clean_panch_df["target_tag"].value_counts().reindex(TARGET_TAGS, fill_value=0)
news_tag_dist = raw_news_df["gold_sanskrit_tag"].value_counts().reindex(TARGET_TAGS, fill_value=0)

dist_df = pd.DataFrame({
    "pos_tag": TARGET_TAGS,
    "panchatantra_count": panch_tag_dist.values,
    "panchatantra_pct": (panch_tag_dist.values / len(clean_panch_df) * 100).round(3),
    "news_count": news_tag_dist.values,
    "news_pct": (news_tag_dist.values / len(raw_news_df) * 100).round(3),
})
dist_df.to_csv(DIRS["evaluations"] / "pos_tag_distribution_comparison.csv", index=False)
print(dist_df.to_string(index=False))

panch_nonzero = panch_tag_dist[panch_tag_dist > 0]
news_nonzero = news_tag_dist[news_tag_dist > 0]
print(f"\nPanchatantra active tags: {len(panch_nonzero)}/24 | Majority: {panch_nonzero.idxmax()} ({panch_nonzero.max()}) | Minority: {panch_nonzero.idxmin()} ({panch_nonzero.min()}) | Imbalance Ratio: {panch_nonzero.max()/panch_nonzero.min():.1f}x")
print(f"News active tags        : {len(news_nonzero)}/24 | Majority: {news_nonzero.idxmax()} ({news_nonzero.max()}) | Minority: {news_nonzero.idxmin()} ({news_nonzero.min()}) | Imbalance Ratio: {news_nonzero.max()/news_nonzero.min():.1f}x")

# Save distribution plots
for col, title, fname, color in [
    ("panchatantra_count", "Panchatantra POS Tag Distribution (Mapped 24 Tags)", "panchatantra_pos_distribution.png", "#2b5c8f"),
    ("news_count", "Sanskrit News POS Tag Distribution (24 Target Tags)", "news_pos_distribution.png", "#c44e52"),
]:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(dist_df["pos_tag"], dist_df[col], color=color)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("POS Tag")
    ax.set_ylabel("Token Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / fname, dpi=200)
    plt.close(fig)

pos_tag  panchatantra_count  panchatantra_pct  news_count  news_pct
      N                9673            22.202       12108    49.829
     NS                 727             1.669        1344     5.531
     NV                3536             8.116        3035    12.490
    KNV                 276             0.633         256     1.054
     SN                4930            11.316        1180     4.856
    SNV                 386             0.886          17     0.070
    SNS                   0             0.000           2     0.008
    SAM                   0             0.000          30     0.123
   SAMY                   0             0.000           6     0.025
   SAMP                   0             0.000          28     0.115
     KP                3848             8.832        2197     9.042
     UP                 117             0.269          34     0.140
    AVN                7734            17.752        2403     9.889
    AVP                 401             0.920   

## 8. Dataset Splitting (70/15/15 Panchatantra + Validated News Splits)
- **Source Domain (Panchatantra):** Split 70% Train / 15% Validation / 15% Held-Out Test (`RANDOM_SEED = 42`), grouping identical sentence texts together so no duplicate sentence appears across splits.
- **Target Domain (News):** Reuse the validated, document-disjoint `news_train_pool.json` (1,484 sentences), `news_validation.json` (370 sentences), and `news_test_locked.json` (791 sentences).

In [8]:
# Group Panchatantra into sentence-level DataFrame
panch_sentences = (
    clean_panch_df.sort_values(["sentence_id", "word_number"])
    .groupby("sentence_id", sort=False)
    .agg({"word": list, "target_tag": list, "tag": list, "sentence": "first"})
    .reset_index()
    .rename(columns={"word": "tokens", "target_tag": "tags", "tag": "source_tags"})
)
panch_sentences["doc_id"] = panch_sentences["sentence_id"]
panch_sentences["sent_text_key"] = panch_sentences["tokens"].apply(lambda t: " ".join(t))

# Group by exact token sequence to prevent duplicate-sentence leakage across splits
unique_texts = panch_sentences["sent_text_key"].unique()
rng = np.random.RandomState(RANDOM_SEED)
perm_texts = rng.permutation(unique_texts)

n_unique = len(perm_texts)
n_train_u = int(0.70 * n_unique)
n_val_u = int(0.15 * n_unique)

train_texts = set(perm_texts[:n_train_u])
val_texts = set(perm_texts[n_train_u : n_train_u + n_val_u])
test_texts = set(perm_texts[n_train_u + n_val_u :])

panch_train = panch_sentences[panch_sentences["sent_text_key"].isin(train_texts)].reset_index(drop=True)
panch_val = panch_sentences[panch_sentences["sent_text_key"].isin(val_texts)].reset_index(drop=True)
panch_test = panch_sentences[panch_sentences["sent_text_key"].isin(test_texts)].reset_index(drop=True)

# Load existing validated News splits
def load_json_split(fname: str) -> pd.DataFrame:
    return pd.DataFrame(json.loads((EXISTING_DATASET_DIR / fname).read_text(encoding="utf-8")))

news_pool = load_json_split("news_train_pool.json")
news_val = load_json_split("news_validation.json")
news_test = load_json_split("news_test_locked.json")

splits = {
    "panchatantra_train": panch_train,
    "panchatantra_validation": panch_val,
    "panchatantra_test": panch_test,
    "news_adaptation_pool": news_pool,
    "news_validation": news_val,
    "news_test_locked": news_test,
}

# Save splits as CSV and JSON under outputs/Panchatantra_output/datasets/
manifest = {"random_seed": RANDOM_SEED, "splits": {}}
for sname, sdf in splits.items():
    tok_count = int(sum(len(t) for t in sdf["tokens"]))
    manifest["splits"][sname] = {
        "sentences": int(len(sdf)),
        "tokens": tok_count,
        "sentence_ids": [str(x) for x in sdf["sentence_id"].tolist()[:20]] + ["..."],
    }
    sdf[["doc_id", "sentence_id", "tokens", "tags"]].to_json(
        DIRS["datasets"] / f"{sname}.json", orient="records", force_ascii=False, indent=2
    )
    # Also save token-level CSV
    rows = []
    for r in sdf.itertuples():
        for idx, (tok, tag) in enumerate(zip(r.tokens, r.tags), 1):
            rows.append({"doc_id": r.doc_id, "sentence_id": r.sentence_id, "token_id": idx, "token": tok, "tag": tag})
    pd.DataFrame(rows).to_csv(DIRS["datasets"] / f"{sname}.csv", index=False)
    print(f"{sname:24s}: {len(sdf):5d} sentences, {tok_count:6d} tokens")

(DIRS["datasets"] / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

panchatantra_train      :  3928 sentences,  30594 tokens
panchatantra_validation :   763 sentences,   6550 tokens
panchatantra_test       :   695 sentences,   6424 tokens
news_adaptation_pool    :  1484 sentences,  13544 tokens
news_validation         :   370 sentences,   3431 tokens
news_test_locked        :   791 sentences,   7324 tokens


2436

## 9. Leakage Verification and Dataset Audit Report

In [9]:
for sname, sdf in splits.items():
    validate_split_frame(sdf, sname)

# Verify zero sentence/text overlap across Panchatantra splits
p_tr_t = set(panch_train["sent_text_key"])
p_va_t = set(panch_val["sent_text_key"])
p_te_t = set(panch_test["sent_text_key"])
assert len(p_tr_t & p_va_t) == 0 and len(p_tr_t & p_te_t) == 0 and len(p_va_t & p_te_t) == 0, "Panchatantra text overlap!"

# Verify zero document/sentence overlap across News splits
validate_split_leakage({
    "news_adaptation_pool": news_pool,
    "news_validation": news_val,
    "news_test_locked": news_test,
})

# Verify zero cross-domain text contamination
news_texts_all = set(news_pool["tokens"].apply(lambda t: " ".join(t))) | set(news_val["tokens"].apply(lambda t: " ".join(t))) | set(news_test["tokens"].apply(lambda t: " ".join(t)))
cross_overlap = (p_tr_t | p_va_t | p_te_t) & news_texts_all
print(f"Panchatantra split overlap: 0 | News doc/sent overlap: 0 | Cross-domain exact overlap: {len(cross_overlap)}")

# Write DATASET_AUDIT_REPORT.md
ds_report = f"""# Dataset Audit Report: Panchatantra & Sanskrit News

**Generated:** {datetime.now(timezone.utc).isoformat()}
**Random Seed:** {RANDOM_SEED}

## 1. Source Domain: `Panchatantra_clean.csv`
- **Raw Tokens:** {len(raw_panch_df):,} across {raw_panch_df['sentence_id'].nunique():,} sentences
- **Excluded Incomplete Sentences:** {len(excluded_sids)} sentences ({len(excluded_df)} tokens containing 6 `NaN` tags and 1 unresolved `UA` tag)
- **Clean Mapped Tokens:** {len(clean_panch_df):,} across {len(panch_sentences):,} sentences ({coverage_pct:.2f}% raw coverage)
- **Token Artifacts:** {len(slash_artifacts)} tokens contain `/` from upstream comma-attached slash tags (preserved transparently)

## 2. Target Domain: `News_Annotated.csv` (Reused Validated Splits)
- **News Adaptation Pool:** {len(news_pool):,} sentences, {sum(len(t) for t in news_pool['tokens']):,} tokens ({news_pool['doc_id'].nunique():,} documents)
- **News Validation:** {len(news_val):,} sentences, {sum(len(t) for t in news_val['tokens']):,} tokens ({news_val['doc_id'].nunique():,} documents)
- **Locked News Test:** {len(news_test):,} sentences, {sum(len(t) for t in news_test['tokens']):,} tokens ({news_test['doc_id'].nunique():,} documents)

## 3. Panchatantra Splits (70% / 15% / 15%, Duplicate-Free)
- **Panchatantra Train:** {len(panch_train):,} sentences, {sum(len(t) for t in panch_train['tokens']):,} tokens
- **Panchatantra Validation:** {len(panch_val):,} sentences, {sum(len(t) for t in panch_val['tokens']):,} tokens
- **Panchatantra Held-Out Test:** {len(panch_test):,} sentences, {sum(len(t) for t in panch_test['tokens']):,} tokens

## 4. Leakage & Integrity Verification
- Zero sentence-text overlap across Panchatantra Train/Val/Test.
- Zero document or sentence overlap across News Pool/Val/Test.
- 100% token-to-tag length parity and 100% `TARGET_TAGS` validity across all 6 splits.
"""
(OUTPUT_ROOT / "DATASET_AUDIT_REPORT.md").write_text(ds_report, encoding="utf-8")
print("Saved DATASET_AUDIT_REPORT.md")

Panchatantra split overlap: 0 | News doc/sent overlap: 0 | Cross-domain exact overlap: 0
Saved DATASET_AUDIT_REPORT.md


## 10. Model Loading, Tokenization, and Forward-Pass Verification

In [10]:
MODEL_SPECS = [
    {
        "model_id": "M1",
        "key": "bilstm_crf",
        "name": "BiLSTM-CRF",
        "type": "bilstm",
        "config_path": PROJECT_ROOT / "configs" / "bilstm_crf.json",
        "is_iast": False,
    },
    {
        "model_id": "M2",
        "key": "xlm_roberta",
        "name": "XLM-RoBERTa",
        "type": "transformer",
        "hf_id": "FacebookAI/xlm-roberta-base",
        "config_path": PROJECT_ROOT / "configs" / "xlm_roberta.json",
        "is_iast": False,
    },
    {
        "model_id": "M3",
        "key": "indicbert",
        "name": "IndicBERT v2",
        "type": "transformer",
        "hf_id": "ai4bharat/IndicBERTv2-MLM-only",
        "config_path": PROJECT_ROOT / "configs" / "indicbert_v2.json",
        "is_iast": False,
    },
    {
        "model_id": "M4",
        "key": "muril",
        "name": "MuRIL",
        "type": "transformer",
        "hf_id": "google/muril-base-cased",
        "config_path": PROJECT_ROOT / "configs" / "muril.json",
        "is_iast": False,
    },
    {
        "model_id": "M5",
        "key": "sanskrit_bert",
        "name": "Sanskrit-BERT",
        "type": "transformer",
        "hf_id": "Matej/bert-base-buddhist-sanskrit-v2",
        "config_path": PROJECT_ROOT / "configs" / "sanskrit_bert.json",
        "is_iast": True,
    },
]

# Build Panchatantra vocabulary for M1 and save configurations
panch_vocab = build_vocabulary(panch_train["tokens"])
print(f"Panchatantra BiLSTM-CRF word vocab size: {len(panch_vocab['word']):,}")

sample_words = list(panch_train["tokens"].iloc[0])[:6]
sample_tags = list(panch_train["tags"].iloc[0])[:6]
print("Sample Sanskrit words:", sample_words)

for spec in MODEL_SPECS:
    cfg = json.loads(spec["config_path"].read_text(encoding="utf-8"))
    spec["config"] = cfg
    (DIRS["configurations"] / f"{spec['key']}_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")

    if spec["type"] == "bilstm":
        m = BiLSTMCRF(
            vocab_size=len(panch_vocab["word"]),
            num_labels=len(TARGET_TAGS),
            embedding_dim=cfg["embedding_dim"],
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
            use_morphology=cfg.get("use_morphology", False),
        ).to(DEVICE)
        ds = SequenceTaggingDataset(panch_train.iloc[:2], panch_vocab)
        batch = next(iter(torch.utils.data.DataLoader(ds, batch_size=2, collate_fn=collate_sequences)))
        loss = m(batch["input_ids"].to(DEVICE), mask=batch["mask"].to(DEVICE), tags=batch["labels"].to(DEVICE))
        preds = m(batch["input_ids"].to(DEVICE), mask=batch["mask"].to(DEVICE))
        assert len(preds[0]) == len(panch_train.iloc[0]["tokens"])
        print(f"[{spec['model_id']}] {spec['name']}: Verified! Loss={loss.item():.4f}, Decoded len={len(preds[0])}")
        del m
    else:
        tagger = TransformerTagger(model_id=spec["hf_id"], is_iast=spec["is_iast"]).to(DEVICE)
        words_in = [devanagari_to_iast(w) for w in sample_words] if spec["is_iast"] else sample_words
        toks = tagger.tokenizer.tokenize(words_in, is_split_into_words=True)
        preds = tagger.predict_words([sample_words], device=DEVICE, batch_size=1)
        assert len(preds[0]) == len(sample_words)
        print(f"[{spec['model_id']}] {spec['name']}: Verified! Subwords={toks[:10]}, Word preds len={len(preds[0])}")
        del tagger
    torch.cuda.empty_cache()

Panchatantra BiLSTM-CRF word vocab size: 8,603
Sample Sanskrit words: ['॥']


[M1] BiLSTM-CRF: Verified! Loss=3.1352, Decoded len=1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/197 [00:00<00:00, 41527.76it/s, Materializing param=roberta.embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/197 [00:00<00:01, 166.39it/s, Materializing param=roberta.embeddings.LayerNorm.bias]  

Loading weights:   1%|          | 2/197 [00:00<00:00, 239.33it/s, Materializing param=roberta.embeddings.LayerNorm.weight]

... (1,590 outputs left out)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1577.76it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1554.66it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]


BertForTokenClassification LOAD REPORT from: Matej/bert-base-buddhist-sanskrit-v2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Cons

[M5] Sanskrit-BERT: Verified! Subwords=['.', '##.'], Word preds len=1


## 11 & 12. Stage 1: Panchatantra In-Domain Training and Evaluation
For each of the five models:
- Train on `panchatantra_train`
- Select best checkpoint on `panchatantra_validation` Macro F1
- Evaluate on `panchatantra_test`
- Save checkpoint, training history, predictions, per-tag metrics, and confusion matrix.

In [11]:
STATUS_FILE = DIRS["experiment_status"] / "status_records.json"

def load_status() -> dict:
    if STATUS_FILE.is_file():
        return json.loads(STATUS_FILE.read_text(encoding="utf-8"))
    return {}

def save_status(status_dict: dict) -> None:
    STATUS_FILE.write_text(json.dumps(status_dict, indent=2), encoding="utf-8")
    # Also write EXPERIMENT_STATUS.md
    lines = [
        "# Panchatantra Experiment Execution Status",
        "",
        f"**Last Updated:** {datetime.now(timezone.utc).isoformat()}",
        "",
        "| Key | Model | Stage | Budget | Seed | Status | Macro F1 | Checkpoint |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for k, v in status_dict.items():
        f1_str = f"{v['macro_f1']:.4f}" if v.get("macro_f1") is not None else "—"
        lines.append(f"| {k} | {v['model']} | {v['stage']} | {v.get('budget', 0)} | {v.get('seed', 42)} | {v['status']} | {f1_str} | `{v.get('checkpoint', '')}` |")
    (OUTPUT_ROOT / "EXPERIMENT_STATUS.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

exp_status = load_status()
stage1_results = []
per_tag_all_rows = []

for spec in MODEL_SPECS:
    mkey = spec["key"]
    mid = spec["model_id"]
    mname = spec["name"]
    cfg = spec["config"]
    run_key = f"{mkey}__stage1_in_domain"
    ckpt_target = DIRS["models"] / mkey / "panchatantra_in_domain"
    eval_dir = DIRS["evaluations"] / f"{mkey}_stage1_in_domain"

    if run_key in exp_status and exp_status[run_key]["status"] == "completed" and (eval_dir / "summary_metrics.json").is_file():
        print(f"[{mid}] {mname} Stage 1 already completed — loading saved metrics.")
        summary = json.loads((eval_dir / "summary_metrics.json").read_text(encoding="utf-8"))
        per_tag = pd.read_csv(eval_dir / "per_tag_metrics.csv")
        stage1_results.append(exp_status[run_key]["record"])
        for r in per_tag.itertuples():
            per_tag_all_rows.append({"model_id": mid, "model_name": mname, "stage": "panchatantra_in_domain", "budget": 0, "seed": RANDOM_SEED, "pos_tag": r.tag, "precision": r.precision, "recall": r.recall, "f1_score": r.f1, "support": r.support})
        continue

    print(f"\n>>> [{mid}] Stage 1: Training {mname} on Panchatantra...")
    set_seed(RANDOM_SEED)
    t_start = time.time()
    try:
        if spec["type"] == "bilstm":
            model = BiLSTMCRF(
                vocab_size=len(panch_vocab["word"]),
                num_labels=len(TARGET_TAGS),
                embedding_dim=cfg["embedding_dim"],
                hidden_size=cfg["hidden_size"],
                num_layers=cfg["num_layers"],
                dropout=cfg["dropout"],
                use_morphology=cfg.get("use_morphology", False),
            ).to(DEVICE)
            best_model, val_metrics, _ = train_bilstm_crf(
                model=model, train_df=panch_train, val_df=panch_val,
                vocabulary=panch_vocab, device=DEVICE, run_dir=ckpt_target, config=cfg
            )
            train_time = time.time() - t_start
            t_inf = time.time()
            test_ds = SequenceTaggingDataset(panch_test, panch_vocab)
            test_loader = torch.utils.data.DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate_sequences)
            test_preds = decode_bilstm_predictions(best_model, test_loader, DEVICE)
            inf_time = time.time() - t_inf
            ckpt_path_str = str(ckpt_target / "checkpoint" / "model.pt")
            del model, best_model
        else:
            tagger = TransformerTagger(model_id=spec["hf_id"], is_iast=spec["is_iast"]).to(DEVICE)
            best_tagger, val_metrics, _ = train_transformer(
                tagger=tagger, train_df=panch_train, val_df=panch_val,
                device=DEVICE, run_dir=ckpt_target, config=cfg
            )
            train_time = time.time() - t_start
            t_inf = time.time()
            test_preds = best_tagger.predict_words(list(panch_test["tokens"]), device=DEVICE, batch_size=cfg["batch_size"], max_length=cfg["max_length"])
            inf_time = time.time() - t_inf
            ckpt_path_str = str(ckpt_target / "checkpoint")
            del tagger, best_tagger

        torch.cuda.empty_cache()
        summary, per_tag, report, conf_mat = evaluate_predictions(panch_test, test_preds)
        save_evaluation_results(eval_dir, summary, per_tag, report, conf_mat, test_preds, panch_test)
        plot_confusion_matrix_heatmap(conf_mat, f"{mname} — Panchatantra In-Domain", DIRS["confusion_matrices"] / f"{mkey}_panchatantra_in_domain_cm.png")

        rec = {
            "model_id": mid, "model_name": mname, "stage": "panchatantra_in_domain",
            "train_dataset": "panchatantra_train", "val_dataset": "panchatantra_validation", "test_dataset": "panchatantra_test",
            "train_sentences": len(panch_train), "val_sentences": len(panch_val), "test_sentences": len(panch_test),
            "accuracy": summary["accuracy"], "macro_precision": summary["macro_precision"], "macro_recall": summary["macro_recall"],
            "macro_f1": summary["macro_f1"], "weighted_precision": summary["weighted_precision"],
            "weighted_recall": summary["weighted_recall"], "weighted_f1": summary["weighted_f1"],
            "training_time_sec": round(train_time, 2), "inference_time_sec": round(inf_time, 2),
            "seed": RANDOM_SEED, "checkpoint_path": ckpt_path_str, "status": "completed",
        }
        stage1_results.append(rec)
        for r in per_tag.itertuples():
            per_tag_all_rows.append({"model_id": mid, "model_name": mname, "stage": "panchatantra_in_domain", "budget": 0, "seed": RANDOM_SEED, "pos_tag": r.tag, "precision": r.precision, "recall": r.recall, "f1_score": r.f1, "support": r.support})

        exp_status[run_key] = {"model": mname, "stage": "panchatantra_in_domain", "budget": 0, "seed": RANDOM_SEED, "status": "completed", "macro_f1": summary["macro_f1"], "checkpoint": ckpt_path_str, "record": rec}
        save_status(exp_status)
        print(f"[{mid}] {mname} Stage 1 Done: Acc={summary['accuracy']:.4f}, Macro F1={summary['macro_f1']:.4f}, Weighted F1={summary['weighted_f1']:.4f}")
    except Exception as e:
        exp_status[run_key] = {"model": mname, "stage": "panchatantra_in_domain", "budget": 0, "seed": RANDOM_SEED, "status": "failed", "error": str(e)}
        save_status(exp_status)
        raise

stage1_df = pd.DataFrame(stage1_results)
stage1_df.to_csv(DIRS["evaluations"] / "panchatantra_in_domain_results.csv", index=False)
print("\n=== Stage 1 Summary ===")
print(stage1_df[["model_id", "model_name", "accuracy", "macro_f1", "weighted_f1", "training_time_sec"]].to_string(index=False))


>>> [M1] Stage 1: Training BiLSTM-CRF on Panchatantra...


[M1] BiLSTM-CRF Stage 1 Done: Acc=0.8202, Macro F1=0.3875, Weighted F1=0.8070

>>> [M2] Stage 1: Training XLM-RoBERTa on Panchatantra...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/197 [00:00<00:00, 62601.55it/s, Materializing param=roberta.embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/197 [00:00<00:00, 1763.05it/s, Materializing param=roberta.embeddings.LayerNorm.bias] 

Loading weights:   1%|          | 2/197 [00:00<00:00, 1796.66it/s, Materializing param=roberta.embeddings.LayerNorm.weight]

... (3,289 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1492.27it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1492.27it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1393.76it/s, Materializing param=classifier.weight]

[M5] Sanskrit-BERT Stage 1 Done: Acc=0.8898, Macro F1=0.4494, Weighted F1=0.8883

=== Stage 1 Summary ===
model_id    model_name  accuracy  macro_f1  weighted_f1  training_time_sec
      M1    BiLSTM-CRF  0.820205  0.387530     0.806971              50.17
      M2   XLM-RoBERTa  0.896171  0.466468     0.895365             454.71
      M3  IndicBERT v2  0.907690  0.491859     0.908606             476.27
      M4         MuRIL  0.901775  0.426768     0.895900             529.85
      M5 Sanskrit-BERT  0.889788  0.449438     0.888299             333.53


## 13. Stage 2: Zero-Shot Cross-Domain Generalization (Panchatantra → Locked News Test)
Each model's Stage 1 Panchatantra checkpoint is evaluated directly on `news_test_locked` with zero news-domain training.
We compute `absolute_f1_drop` and `relative_f1_drop` relative to Panchatantra in-domain Macro F1.

In [12]:
stage2_results = []
drop_results = []

for spec in MODEL_SPECS:
    mkey = spec["key"]
    mid = spec["model_id"]
    mname = spec["name"]
    cfg = spec["config"]
    run_key = f"{mkey}__stage2_zero_shot"
    s1_rec = next(r for r in stage1_results if r["model_id"] == mid)
    ckpt_path = Path(s1_rec["checkpoint_path"])
    eval_dir = DIRS["evaluations"] / f"{mkey}_stage2_zero_shot"

    if run_key in exp_status and exp_status[run_key]["status"] == "completed" and (eval_dir / "summary_metrics.json").is_file():
        print(f"[{mid}] {mname} Stage 2 already completed — loading saved metrics.")
        rec = exp_status[run_key]["record"]
        stage2_results.append(rec)
        per_tag = pd.read_csv(eval_dir / "per_tag_metrics.csv")
        for r in per_tag.itertuples():
            per_tag_all_rows.append({"model_id": mid, "model_name": mname, "stage": "zero_shot", "budget": 0, "seed": RANDOM_SEED, "pos_tag": r.tag, "precision": r.precision, "recall": r.recall, "f1_score": r.f1, "support": r.support})
    else:
        print(f"\n>>> [{mid}] Stage 2: Zero-Shot Evaluation ({mname})...")
        t_inf = time.time()
        if spec["type"] == "bilstm":
            lit_m, vocab, _ = BiLSTMCRF.load(ckpt_path, device=DEVICE)
            test_ds = SequenceTaggingDataset(news_test, vocab)
            test_loader = torch.utils.data.DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate_sequences)
            test_preds = decode_bilstm_predictions(lit_m, test_loader, DEVICE)
            del lit_m
        else:
            tagger = TransformerTagger.load(ckpt_path, device=DEVICE)
            test_preds = tagger.predict_words(list(news_test["tokens"]), device=DEVICE, batch_size=cfg["batch_size"], max_length=cfg["max_length"])
            del tagger
        inf_time = time.time() - t_inf
        torch.cuda.empty_cache()

        summary, per_tag, report, conf_mat = evaluate_predictions(news_test, test_preds)
        save_evaluation_results(eval_dir, summary, per_tag, report, conf_mat, test_preds, news_test)
        plot_confusion_matrix_heatmap(conf_mat, f"{mname} — Zero-Shot (Panchatantra -> News)", DIRS["confusion_matrices"] / f"{mkey}_zero_shot_cm.png")

        rec = {
            "model_id": mid, "model_name": mname, "stage": "zero_shot",
            "source_domain": "Panchatantra", "target_domain": "Sanskrit News (Locked Test)",
            "accuracy": summary["accuracy"], "macro_precision": summary["macro_precision"], "macro_recall": summary["macro_recall"],
            "macro_f1": summary["macro_f1"], "weighted_precision": summary["weighted_precision"],
            "weighted_recall": summary["weighted_recall"], "weighted_f1": summary["weighted_f1"],
            "inference_time_sec": round(inf_time, 2), "seed": RANDOM_SEED,
            "checkpoint_path": str(ckpt_path), "status": "completed",
        }
        stage2_results.append(rec)
        for r in per_tag.itertuples():
            per_tag_all_rows.append({"model_id": mid, "model_name": mname, "stage": "zero_shot", "budget": 0, "seed": RANDOM_SEED, "pos_tag": r.tag, "precision": r.precision, "recall": r.recall, "f1_score": r.f1, "support": r.support})

        exp_status[run_key] = {"model": mname, "stage": "zero_shot", "budget": 0, "seed": RANDOM_SEED, "status": "completed", "macro_f1": summary["macro_f1"], "checkpoint": str(ckpt_path), "record": rec}
        save_status(exp_status)

    abs_drop = s1_rec["macro_f1"] - rec["macro_f1"]
    rel_drop = (abs_drop / s1_rec["macro_f1"] * 100.0) if s1_rec["macro_f1"] > 0 else 0.0
    drop_results.append({
        "model_id": mid, "model_name": mname,
        "panchatantra_in_domain_macro_f1": s1_rec["macro_f1"],
        "zero_shot_news_macro_f1": rec["macro_f1"],
        "absolute_f1_drop_pp": round(abs_drop * 100, 2),
        "relative_f1_drop_pct": round(rel_drop, 2),
        "panchatantra_accuracy": s1_rec["accuracy"],
        "zero_shot_accuracy": rec["accuracy"],
    })
    print(f"[{mid}] {mname} Zero-Shot: Macro F1={rec['macro_f1']:.4f} | Drop={abs_drop*100:.2f} pp ({rel_drop:.2f}% relative)")

stage2_df = pd.DataFrame(stage2_results)
drop_df = pd.DataFrame(drop_results)
stage2_df.to_csv(DIRS["evaluations"] / "zero_shot_cross_domain_results.csv", index=False)
drop_df.to_csv(DIRS["evaluations"] / "performance_drop_results.csv", index=False)


>>> [M1] Stage 2: Zero-Shot Evaluation (BiLSTM-CRF)...


[M1] BiLSTM-CRF Zero-Shot: Macro F1=0.1514 | Drop=23.62 pp (60.94% relative)

>>> [M2] Stage 2: Zero-Shot Evaluation (XLM-RoBERTa)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/199 [00:00<00:00, 51781.53it/s, Materializing param=classifier.bias]

Loading weights:   1%|          | 1/199 [00:00<00:00, 889.00it/s, Materializing param=classifier.bias]  

Loading weights:   1%|          | 2/199 [00:00<00:00, 1029.02it/s, Materializing param=classifier.weight]

... (1,600 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1417.08it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1417.08it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1356.38it/s, Materializing param=classifier.weight]

[M5] Sanskrit-BERT Zero-Shot: Macro F1=0.1988 | Drop=25.06 pp (55.76% relative)


## 14. Stage 3 Prep: Strictly Nested Few-Shot Subset Sampling
We sample nested subsets (`50 ⊂ 100 ⊂ 200 ⊂ 500 ⊂ 1000`) exclusively from `news_adaptation_pool` (1,484 sentences) across seeds `[42, 123, 2026]` and save exact sentence/doc IDs and token counts.

In [13]:
nested_sample_meta = []
for s in FEW_SHOT_SEEDS:
    subsets = get_nested_few_shot_subsets(news_pool, budgets=FEW_SHOT_BUDGETS, seed=s)
    prev_ids = set()
    for b in FEW_SHOT_BUDGETS:
        sub = subsets[b]
        cur_ids = set(zip(sub["doc_id"], sub["sentence_id"]))
        assert prev_ids.issubset(cur_ids), f"Nesting violated at seed={s}, budget={b}"
        prev_ids = cur_ids
        tok_count = int(sum(len(t) for t in sub["tokens"]))
        sub[["doc_id", "sentence_id"]].to_csv(DIRS["few_shot_samples"] / f"few_shot_ids_b{b}_s{s}.csv", index=False)
        nested_sample_meta.append({"seed": s, "budget": b, "sentences": len(sub), "tokens": tok_count})

pd.DataFrame(nested_sample_meta).to_csv(DIRS["few_shot_samples"] / "few_shot_subset_summary.csv", index=False)
print(pd.DataFrame(nested_sample_meta).head(10).to_string(index=False))

 seed  budget  sentences  tokens
   42      50         50     485
   42     100        100     939
   42     200        200    1844
   42     500        500    4505
   42    1000       1000    9224
  123      50         50     455
  123     100        100     933
  123     200        200    1916
  123     500        500    4530
  123    1000       1000    9178


## 15 & 16. Stage 3: Independent Few-Shot Adaptation & Locked News Test Evaluation
For every model, budget (`[50, 100, 200, 500, 1000]`), and seed (`[42, 123, 2026]`):
- Independently load the original Stage 1 Panchatantra checkpoint.
- Fine-tune on the nested news subset with early stopping on `news_validation` Macro F1.
- Evaluate on `news_test_locked` and compute improvement over zero-shot (percentage points).

In [14]:
stage3_results = []

# Include 0-sentence zero-shot baseline row for each model
for rec2 in stage2_results:
    stage3_results.append({
        "model_id": rec2["model_id"], "model_name": rec2["model_name"],
        "annotation_budget": 0, "seed": RANDOM_SEED,
        "annotated_sentences": 0, "annotated_tokens": 0,
        "accuracy": rec2["accuracy"], "macro_precision": rec2["macro_precision"],
        "macro_recall": rec2["macro_recall"], "macro_f1": rec2["macro_f1"],
        "weighted_precision": rec2["weighted_precision"], "weighted_recall": rec2["weighted_recall"],
        "weighted_f1": rec2["weighted_f1"], "zero_shot_macro_f1": rec2["macro_f1"],
        "improvement_pp": 0.0, "training_time_sec": 0.0, "inference_time_sec": rec2["inference_time_sec"],
        "checkpoint_path": rec2["checkpoint_path"], "status": "completed",
    })

for spec in MODEL_SPECS:
    mkey = spec["key"]
    mid = spec["model_id"]
    mname = spec["name"]
    cfg = spec["config"]
    s1_rec = next(r for r in stage1_results if r["model_id"] == mid)
    s2_rec = next(r for r in stage2_results if r["model_id"] == mid)
    panch_ckpt = Path(s1_rec["checkpoint_path"])
    zs_f1 = s2_rec["macro_f1"]

    for s in FEW_SHOT_SEEDS:
        subsets = get_nested_few_shot_subsets(news_pool, budgets=FEW_SHOT_BUDGETS, seed=s)
        for b in FEW_SHOT_BUDGETS:
            sub_df = subsets[b]
            tok_count = int(sum(len(t) for t in sub_df["tokens"]))
            run_key = f"{mkey}__few_shot_b{b}_s{s}"
            ckpt_target = DIRS["models"] / mkey / f"few_shot_b{b}_s{s}"
            eval_dir = DIRS["evaluations"] / f"{mkey}_few_shot_b{b}_s{s}"

            if run_key in exp_status and exp_status[run_key]["status"] == "completed" and (eval_dir / "summary_metrics.json").is_file():
                rec = exp_status[run_key]["record"]
                stage3_results.append(rec)
                per_tag = pd.read_csv(eval_dir / "per_tag_metrics.csv")
                for r in per_tag.itertuples():
                    per_tag_all_rows.append({"model_id": mid, "model_name": mname, "stage": f"few_shot_{b}", "budget": b, "seed": s, "pos_tag": r.tag, "precision": r.precision, "recall": r.recall, "f1_score": r.f1, "support": r.support})
                continue

            print(f">>> [{mid}] {mname} | Budget={b}, Seed={s}...")
            set_seed(s)
            t_start = time.time()
            ft_cfg = copy.deepcopy(cfg)
            if spec["type"] == "bilstm":
                ft_cfg["learning_rate"] = cfg.get("learning_rate", 0.001) * 0.5
                ft_cfg["epochs"] = 15
                ft_model, ft_vocab, _ = BiLSTMCRF.load(panch_ckpt, device=DEVICE)
                best_ft, _, _ = train_bilstm_crf(
                    model=ft_model, train_df=sub_df, val_df=news_val,
                    vocabulary=ft_vocab, device=DEVICE, run_dir=ckpt_target, config=ft_cfg
                )
                train_time = time.time() - t_start
                t_inf = time.time()
                test_ds = SequenceTaggingDataset(news_test, ft_vocab)
                test_loader = torch.utils.data.DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate_sequences)
                test_preds = decode_bilstm_predictions(best_ft, test_loader, DEVICE)
                inf_time = time.time() - t_inf
                ckpt_str = str(ckpt_target / "checkpoint" / "model.pt")
                del ft_model, best_ft
            else:
                ft_cfg["learning_rate"] = cfg.get("learning_rate", 3e-5) * 0.7
                ft_cfg["epochs"] = 6
                ft_cfg["early_stopping_patience"] = 2
                ft_tagger = TransformerTagger.load(panch_ckpt, device=DEVICE)
                best_ft, _, _ = train_transformer(
                    tagger=ft_tagger, train_df=sub_df, val_df=news_val,
                    device=DEVICE, run_dir=ckpt_target, config=ft_cfg
                )
                train_time = time.time() - t_start
                t_inf = time.time()
                test_preds = best_ft.predict_words(list(news_test["tokens"]), device=DEVICE, batch_size=cfg["batch_size"], max_length=cfg["max_length"])
                inf_time = time.time() - t_inf
                ckpt_str = str(ckpt_target / "checkpoint")
                del ft_tagger, best_ft

            torch.cuda.empty_cache()
            summary, per_tag, report, conf_mat = evaluate_predictions(news_test, test_preds)
            save_evaluation_results(eval_dir, summary, per_tag, report, conf_mat, test_preds, news_test)
            if s == 42 and b == 1000:
                plot_confusion_matrix_heatmap(conf_mat, f"{mname} — Few-Shot 1000 (Seed 42)", DIRS["confusion_matrices"] / f"{mkey}_few_shot_1000_cm.png")

            imp_pp = (summary["macro_f1"] - zs_f1) * 100.0
            rec = {
                "model_id": mid, "model_name": mname,
                "annotation_budget": b, "seed": s,
                "annotated_sentences": len(sub_df), "annotated_tokens": tok_count,
                "accuracy": summary["accuracy"], "macro_precision": summary["macro_precision"],
                "macro_recall": summary["macro_recall"], "macro_f1": summary["macro_f1"],
                "weighted_precision": summary["weighted_precision"], "weighted_recall": summary["weighted_recall"],
                "weighted_f1": summary["weighted_f1"], "zero_shot_macro_f1": zs_f1,
                "improvement_pp": round(imp_pp, 2), "training_time_sec": round(train_time, 2),
                "inference_time_sec": round(inf_time, 2), "checkpoint_path": ckpt_str, "status": "completed",
            }
            stage3_results.append(rec)
            for r in per_tag.itertuples():
                per_tag_all_rows.append({"model_id": mid, "model_name": mname, "stage": f"few_shot_{b}", "budget": b, "seed": s, "pos_tag": r.tag, "precision": r.precision, "recall": r.recall, "f1_score": r.f1, "support": r.support})

            exp_status[run_key] = {"model": mname, "stage": "few_shot", "budget": b, "seed": s, "status": "completed", "macro_f1": summary["macro_f1"], "checkpoint": ckpt_str, "record": rec}
            save_status(exp_status)
            print(f"    -> Acc={summary['accuracy']:.4f}, Macro F1={summary['macro_f1']:.4f} (+{imp_pp:.2f} pp)")

few_shot_df = pd.DataFrame(stage3_results)
few_shot_df.to_csv(DIRS["evaluations"] / "few_shot_results.csv", index=False)

>>> [M1] BiLSTM-CRF | Budget=50, Seed=42...


    -> Acc=0.6415, Macro F1=0.1853 (+3.39 pp)
>>> [M1] BiLSTM-CRF | Budget=100, Seed=42...


    -> Acc=0.6335, Macro F1=0.1796 (+2.82 pp)
>>> [M1] BiLSTM-CRF | Budget=200, Seed=42...


    -> Acc=0.6505, Macro F1=0.1905 (+3.91 pp)
>>> [M1] BiLSTM-CRF | Budget=500, Seed=42...


    -> Acc=0.6599, Macro F1=0.1922 (+4.08 pp)
>>> [M1] BiLSTM-CRF | Budget=1000, Seed=42...


    -> Acc=0.6640, Macro F1=0.1946 (+4.32 pp)
>>> [M1] BiLSTM-CRF | Budget=50, Seed=123...


... (49,018 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1508.11it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1508.11it/s, Materializing param=classifier.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1590.81it/s, Materializing param=classifier.weight]

    -> Acc=0.7590, Macro F1=0.3189 (+12.00 pp)


## 17. Master Results & Per-Tag Table Generation

In [15]:
per_tag_df = pd.DataFrame(per_tag_all_rows)
per_tag_df.to_csv(DIRS["evaluations"] / "per_tag_results.csv", index=False)

# Build Master Results Table
master_rows = []
for r in stage1_results:
    master_rows.append({
        "model_id": r["model_id"], "model_name": r["model_name"], "stage": "Stage 1: Panchatantra In-Domain",
        "train_dataset": "panchatantra_train", "val_dataset": "panchatantra_validation", "test_dataset": "panchatantra_test",
        "few_shot_budget": 0, "seed": r["seed"], "accuracy": r["accuracy"],
        "macro_precision": r["macro_precision"], "macro_recall": r["macro_recall"], "macro_f1": r["macro_f1"],
        "weighted_precision": r["weighted_precision"], "weighted_recall": r["weighted_recall"], "weighted_f1": r["weighted_f1"],
        "training_time_sec": r["training_time_sec"], "inference_time_sec": r["inference_time_sec"],
        "delta_vs_zero_shot_pp": None, "status": r["status"],
    })
for r in stage2_results:
    drop_r = next(d for d in drop_results if d["model_id"] == r["model_id"])
    master_rows.append({
        "model_id": r["model_id"], "model_name": r["model_name"], "stage": "Stage 2: Zero-Shot Cross-Domain",
        "train_dataset": "panchatantra_train", "val_dataset": "panchatantra_validation", "test_dataset": "news_test_locked",
        "few_shot_budget": 0, "seed": r["seed"], "accuracy": r["accuracy"],
        "macro_precision": r["macro_precision"], "macro_recall": r["macro_recall"], "macro_f1": r["macro_f1"],
        "weighted_precision": r["weighted_precision"], "weighted_recall": r["weighted_recall"], "weighted_f1": r["weighted_f1"],
        "training_time_sec": 0.0, "inference_time_sec": r["inference_time_sec"],
        "delta_vs_zero_shot_pp": -drop_r["absolute_f1_drop_pp"], "status": r["status"],
    })
for r in stage3_results:
    if r["annotation_budget"] == 0:
        continue
    master_rows.append({
        "model_id": r["model_id"], "model_name": r["model_name"], "stage": f"Stage 3: Few-Shot ({r['annotation_budget']})",
        "train_dataset": f"news_pool_{r['annotation_budget']}", "val_dataset": "news_validation", "test_dataset": "news_test_locked",
        "few_shot_budget": r["annotation_budget"], "seed": r["seed"], "accuracy": r["accuracy"],
        "macro_precision": r["macro_precision"], "macro_recall": r["macro_recall"], "macro_f1": r["macro_f1"],
        "weighted_precision": r["weighted_precision"], "weighted_recall": r["weighted_recall"], "weighted_f1": r["weighted_f1"],
        "training_time_sec": r["training_time_sec"], "inference_time_sec": r["inference_time_sec"],
        "delta_vs_zero_shot_pp": r["improvement_pp"], "status": r["status"],
    })

master_df = pd.DataFrame(master_rows)
master_df.to_csv(DIRS["evaluations"] / "master_results.csv", index=False)

# Aggregate 3-seed summary for few-shot
fs_agg = (
    few_shot_df.groupby(["model_id", "model_name", "annotation_budget"], as_index=False)
    .agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        weighted_f1_mean=("weighted_f1", "mean"),
        improvement_pp_mean=("improvement_pp", "mean"),
    )
)
fs_agg["macro_f1_std"] = fs_agg["macro_f1_std"].fillna(0.0)
fs_agg.to_csv(DIRS["evaluations"] / "few_shot_summary_mean_std.csv", index=False)
print("=== Few-Shot Summary (Mean ± Std across seeds) ===")
print(fs_agg.to_string(index=False))

=== Few-Shot Summary (Mean ± Std across seeds) ===
model_id    model_name  annotation_budget  macro_f1_mean  macro_f1_std  accuracy_mean  weighted_f1_mean  improvement_pp_mean
      M1    BiLSTM-CRF                  0       0.151350      0.000000       0.510240          0.490706             0.000000
      M1    BiLSTM-CRF                 50       0.188671      0.002943       0.638631          0.580907             3.730000
      M1    BiLSTM-CRF                100       0.187493      0.007034       0.640952          0.582552             3.610000
      M1    BiLSTM-CRF                200       0.190707      0.001930       0.651329          0.587106             3.936667
      M1    BiLSTM-CRF                500       0.192459      0.000332       0.662844          0.595013             4.110000
      M1    BiLSTM-CRF               1000       0.195344      0.002486       0.666212          0.599445             4.400000
      M2   XLM-RoBERTa                  0       0.237368      0.000000    

## 18. Comparison Graphs and Visualizations

In [16]:
# 1. In-Domain vs Zero-Shot Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(drop_df))
w = 0.35
ax.bar(x - w/2, drop_df["panchatantra_in_domain_macro_f1"] * 100, w, label="Panchatantra In-Domain", color="#2b5c8f")
ax.bar(x + w/2, drop_df["zero_shot_news_macro_f1"] * 100, w, label="Zero-Shot News", color="#dd8452")
ax.set_xticks(x)
ax.set_xticklabels(drop_df["model_name"], fontsize=11)
ax.set_ylabel("Macro F1 (%)")
ax.set_title("Objective 1: Panchatantra In-Domain vs. Zero-Shot Cross-Domain News Macro F1", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "in_domain_vs_zero_shot.png", dpi=200)
plt.close(fig)

# 2. Few-Shot Learning Curves (Mean ± Std)
fig, ax = plt.subplots(figsize=(10, 6))
for mname, grp in fs_agg.groupby("model_name", sort=False):
    grp = grp.sort_values("annotation_budget")
    ax.errorbar(
        grp["annotation_budget"], grp["macro_f1_mean"] * 100, yerr=grp["macro_f1_std"] * 100,
        marker="o", linewidth=2, capsize=4, label=mname
    )
ax.set_xlabel("Annotated News Sentences (0 = Zero-Shot Baseline)", fontsize=11)
ax.set_ylabel("Locked News Test Macro F1 (%)", fontsize=11)
ax.set_title("Objective 2: Few-Shot Adaptation Learning Curves (Panchatantra → News)", fontweight="bold")
ax.set_xticks([0, 50, 100, 200, 500, 1000])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "few_shot_learning_curves.png", dpi=200)
plt.close(fig)

# 3. Per-Tag F1 Comparison (Zero-Shot vs Few-Shot 1000 for Seed 42)
pt_sub = per_tag_df[(per_tag_df["stage"].isin(["panchatantra_in_domain", "zero_shot", "few_shot_1000"])) & (per_tag_df["seed"] == 42)]
pt_pivot = pt_sub.groupby(["pos_tag", "stage"])["f1_score"].mean().unstack(fill_value=0) * 100
fig, ax = plt.subplots(figsize=(14, 6))
pt_pivot.reindex(TARGET_TAGS).plot(kind="bar", ax=ax, width=0.8)
ax.set_title("Average Per-Tag F1 Across Five Models: In-Domain vs Zero-Shot vs Few-Shot (1000)", fontweight="bold")
ax.set_ylabel("Mean F1 (%)")
ax.set_xlabel("POS Tag")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "per_tag_f1_comparison.png", dpi=200)
plt.close(fig)
print("Saved all comparison figures to", DIRS["figures"])

Saved all comparison figures to /home/siba/P25DS023/Cross-Domain-23/outputs/Panchatantra_output/figures


## 19. Final Research Report Generation

In [17]:
def df_to_md(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for row in df.itertuples(index=False, name=None):
        vals = [f"{v:.4f}" if isinstance(v, float) else str(v) for v in row]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

s1_tbl = stage1_df[["model_id", "model_name", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "training_time_sec", "inference_time_sec"]]
drop_tbl = drop_df[["model_id", "model_name", "panchatantra_in_domain_macro_f1", "zero_shot_news_macro_f1", "absolute_f1_drop_pp", "relative_f1_drop_pct", "panchatantra_accuracy", "zero_shot_accuracy"]]
fs_tbl = fs_agg.copy()

best_zs = drop_df.loc[drop_df["zero_shot_news_macro_f1"].idxmax()]
best_fs1000 = fs_agg[fs_agg["annotation_budget"] == 1000].sort_values("macro_f1_mean", ascending=False).iloc[0]

report_md = f"""# Panchatantra-Based Cross-Domain Generalization and Few-Shot Adaptation for Sanskrit POS Tagging

**Generated:** {datetime.now(timezone.utc).isoformat()}  
**Source Domain:** `Panchatantra_clean.csv` ({len(panch_train):,} train / {len(panch_val):,} val / {len(panch_test):,} test sentences)  
**Target Domain:** `News_Annotated.csv` ({len(news_pool):,} adaptation pool / {len(news_val):,} val / {len(news_test):,} locked test sentences)  
**Target Tagset:** 24 Sanskrit POS tags (`{', '.join(TARGET_TAGS)}`)

---

## 1. Objective 1: Cross-Domain Generalization (Panchatantra → News)

### Stage 1: Panchatantra In-Domain Performance
{df_to_md(s1_tbl)}

### Stage 2: Zero-Shot Cross-Domain Degradation
{df_to_md(drop_tbl)}

**Key Finding (Objective 1):**  
All five models experience substantial cross-domain degradation when evaluated zero-shot on Sanskrit News without target-domain fine-tuning. The strongest zero-shot model is **{best_zs['model_name']}** (Macro F1 = {best_zs['zero_shot_news_macro_f1']:.4f}, Accuracy = {best_zs['zero_shot_accuracy']:.4f}).

---

## 2. Objective 2: Few-Shot Cross-Domain Adaptation

### Stage 3: Few-Shot Adaptation Summary (Mean ± Std across seeds 42, 123, 2026)
{df_to_md(fs_tbl)}

**Key Finding (Objective 2):**  
Independent few-shot fine-tuning from each model's Panchatantra checkpoint consistently improves News POS tagging performance as annotated news sentences increase from 50 to 1,000. At 1,000 sentences, **{best_fs1000['model_name']}** achieves a mean Macro F1 of **{best_fs1000['macro_f1_mean']:.4f}** (+{best_fs1000['improvement_pp_mean']:.2f} pp over zero-shot).

---

## 3. Reproducibility & Artifacts
- **Primary Notebook:** `notebooks/Sanskrit_Panchatantra_CrossDomain_Experiments.ipynb`
- **All Outputs:** `outputs/Panchatantra_output/`
- **Evaluations:** `evaluations/panchatantra_in_domain_results.csv`, `zero_shot_cross_domain_results.csv`, `few_shot_results.csv`, `per_tag_results.csv`, `performance_drop_results.csv`, `master_results.csv`
- **Figures:** `figures/panchatantra_pos_distribution.png`, `news_pos_distribution.png`, `in_domain_vs_zero_shot.png`, `few_shot_learning_curves.png`, `per_tag_f1_comparison.png`, `figures/confusion_matrices/`
"""

(OUTPUT_ROOT / "PANCHATANTRA_FINAL_EXPERIMENT_REPORT.md").write_text(report_md, encoding="utf-8")
print("Saved PANCHATRA_FINAL_EXPERIMENT_REPORT.md successfully!")

Saved PANCHATRA_FINAL_EXPERIMENT_REPORT.md successfully!
